In [105]:
import os
import pickle
import numpy as np
from tqdm import tqdm

EPOS_CACHE = {}
OPT_ROUTES = {}
CACHE_FILE = 'epos_cache.pkl'
ROUTE_FILE = 'Routes_cache.pkl'

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'rb') as f:
        EPOS_CACHE = pickle.load(f)
    print(f"Загружено {len(EPOS_CACHE)} результатов EPOS из кэша")

if os.path.exists(ROUTE_FILE):
    with open(ROUTE_FILE, 'rb') as f:
        OPT_ROUTES = pickle.load(f)
    print(f"Загружено {len(OPT_ROUTES)} маршрутов из кэша")


def save_cache():
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(EPOS_CACHE, f)
    with open(ROUTE_FILE, 'wb') as f:
        pickle.dump(OPT_ROUTES, f)
    print(f"Кэш сохранён: {len(EPOS_CACHE)} полигонов")

def explore_EPOS_cached(agent_id, polygon, client_config, model, num_iter=300):
    if polygon in EPOS_CACHE:
        print(f"   [Кэш] {polygon}")
        return EPOS_CACHE[polygon], OPT_ROUTES.get(polygon, None)

    print(f"   [Обработка] {polygon}...")

    client_prefs = generate_polygon_config(polygon, AUG_global_map, n_clusters=8)

    result, path = run_TORL(
        agent_id=agent_id,
        client_config=client_config,
        client_prefs=client_prefs,
        model=model,
        max_iter=num_iter,
        fitted_map=AUG_global_map[polygon]['distance matrix']
    )

    EPOS_CACHE[polygon] = result
    if path is not None:
        OPT_ROUTES[polygon] = path

    if len(EPOS_CACHE) % 1 == 0:
        save_cache()

    return result, path

def run_EPOS_for_courier(courier_id, dynamic_config, model):
    results = EPOS_CACHE.copy()
    opt_routes = OPT_ROUTES.copy()

    all_polygons = get_all_epos_polygons(EPOS)
    exclude = {
        "AUG--AUG-G-85-4-20-3",
        "AUG--AUG-G-72-14-66-1",
        "AUG--G-424-1-269",
        "AUG--G-249-2-327",
        "AUG--G-231-1-296",
        "AUG--G-65-1-340"
    }
    
    all_polygons = [p for p in all_polygons if p not in exclude]
    
    polygons_to_process = [p for p in all_polygons if p not in results]

    for polygon in tqdm(polygons_to_process, desc=f"Курьер {courier_id}"):
        try:
            polygon_Fid = get_basic_poly_ID(polygon)

            client_config = {
                'client_id': polygon_Fid,
                'polygon_id': polygon,
                'orders': AUG_global_map[polygon]['full map'],
                'time_window': dynamic_config['time_window'],
                'U': dynamic_config['U'],
                'price': dynamic_config['price'],
                'CT_SA': dynamic_config['CT_SA']
            }

            res, path = explore_EPOS_cached(courier_id, polygon, client_config, model, num_iter=250)

            results[polygon] = res
            if path is not None:
                opt_routes[polygon] = path

        except Exception as e:
            print(f"ERROR в полигоне {polygon}: {e}")

    save_cache()
    print(f"Курьер {courier_id} завершён. Всего полигонов: {len(results)}\n")

    return results

model, _ = load_grand_courier_model(device='cpu')

dynamic_config = {
    'U': 900,
    'price': 500,
    'CT_SA': 2,
    'time_window': (0.0, 43000.0)
}

EPOS_results = run_EPOS_for_courier(courier_id=1, dynamic_config=dynamic_config, model=model)

Loaded model (800 samples, val_loss=-2.7171)


Курьер 1:   0%|                              | 0/10 [00:00<?, ?it/s]

   [Обработка] AUG--G-158-1-325...
Всего заказов: 45
Processing batch 1: 9 orders from AUG-G-158-1-325--0-14042
 [SUMO] Running simulation for 10 points...
Step #43200.00 (8ms ~= 125.00*RT, ~501625.00UPS, vehicles TOT 35352 ACT 4013 BUF 25810)   
✓ Batch 1 сохранён с points (размер 10)
Traffic correction applied | Mean multiplier: 0.610
Chosen: branch_and_cut | Policy: [0.2, 0.381, 0.418, 0.0]
   Батч 2 содержит только 2 заказов — добираем до 3
Processing batch 2: 2 orders from AUG-G-158-1-325--0-1820
 [SUMO] Running simulation for 3 points...
Step #43200.00 (12ms ~= 83.33*RT, ~333500.00UPS, vehicles TOT 35351 ACT 4002 BUF 25727)   
✓ Batch 2 сохранён с points (размер 3)
Traffic correction applied | Mean multiplier: 0.333
Chosen: christofides | Policy: [0.225, 0.38, 0.394, 0.0]
Processing batch 3: 12 orders from AUG-G-158-1-325--0-9206
 [SUMO] Running simulation for 13 points...
Step #43200.00 (9ms ~= 111.11*RT, ~446777.78UPS, vehicles TOT 35302 ACT 4021 BUF 25926)   
✓ Batch 3 сохранё

Курьер 1:  10%|█▊                | 1/10 [22:48<3:25:16, 1368.50s/it]

Step #43200.00 (14ms ~= 71.43*RT, ~283785.71UPS, vehicles TOT 35340 ACT 3973 BUF 25788)   
 ✓ Сохранено 4 connecting edges
Маршрут завершён. Всего точек: 45
Кэш сохранён: 1 полигонов
   [Обработка] AUG--G-430-2-287...
Всего заказов: 15
Processing batch 1: 13 orders from AUG-G-430-2-287--0-2260
 [SUMO] Running simulation for 14 points...
Step #43200.00 (9ms ~= 111.11*RT, ~692555.56UPS, vehicles TOT 22807 ACT 6233 BUF 38447)   
✓ Batch 1 сохранён с points (размер 14)
Traffic correction applied | Mean multiplier: 1.995
Chosen: lkh3 | Policy: [0.163, 0.379, 0.458, 0.0]
   Батч 2 содержит только 2 заказов — добираем до 3
Processing batch 2: 1 orders from AUG-G-430-2-287--0-9866
 [SUMO] Running simulation for 2 points...
Step #43200.00 (13ms ~= 76.92*RT, ~480307.69UPS, vehicles TOT 22628 ACT 6244 BUF 38446)   
✓ Batch 2 сохранён с points (размер 2)
Traffic correction applied | Mean multiplier: 0.500
Chosen: branch_and_cut | Policy: [0.198, 0.382, 0.42, 0.0]
[AUG--G-430-2-287] Поиск недостающ

Курьер 1:  20%|███▊               | 2/10 [32:10<1:59:14, 894.29s/it]

... (13 outputs left out)

Курьер 1:  90%|████████████████▏ | 9/10 [2:43:49<18:24, 1104.50s/it]

Step #43200.00 (3ms ~= 333.33*RT, ~345000.00UPS, vehicles TOT 17880 ACT 1035 BUF 10584)   
 ✓ Сохранено 3 connecting edges
Маршрут завершён. Всего точек: 60
Кэш сохранён: 9 полигонов
   [Обработка] AUG--G-94-1-125...
Всего заказов: 30
Processing batch 1: 15 orders from AUG-G-94-1-125--0-6698
 [SUMO] Running simulation for 16 points...
Step #43200.00 (10ms ~= 100.00*RT, ~327500.00UPS, vehicles TOT 26733 ACT 3275 BUF 34579)  
✓ Batch 1 сохранён с points (размер 16)
Traffic correction applied | Mean multiplier: 4.629
Chosen: christofides | Policy: [0.145, 0.37, 0.485, 0.0]
Processing batch 2: 11 orders from AUG-G-94-1-125--0-11878
 [SUMO] Running simulation for 12 points...
Step #43200.00 (6ms ~= 166.67*RT, ~551333.33UPS, vehicles TOT 26530 ACT 3308 BUF 34674)   
✓ Batch 2 сохранён с points (размер 12)
Traffic correction applied | Mean multiplier: 0.875
Chosen: lkh3 | Policy: [0.151, 0.375, 0.474, 0.0]
Processing batch 3: 3 orders from AUG-G-94-1-125--0-11148
 [SUMO] Running simulation fo

Курьер 1: 100%|█████████████████| 10/10 [2:53:32<00:00, 1041.24s/it]

Step #43200.00 (9ms ~= 111.11*RT, ~363333.33UPS, vehicles TOT 26399 ACT 3270 BUF 34675)   
 ✓ Сохранено 1 connecting edges
Маршрут завершён. Всего точек: 30
Кэш сохранён: 10 полигонов
Кэш сохранён: 10 полигонов
Курьер 1 завершён. Всего полигонов: 10



In [107]:
def compare_routes_with_initial():
    print("СРАВНЕНИЕ ИСХОДНЫХ И ОПТИМИЗИРОВАННЫХ МАРШРУТОВ")
    
    all_polygons = sorted(EPOS_CACHE.keys())
    
    for polygon_id in all_polygons:
        orders = AUG_global_map[polygon_id].get('full map', [])
        n_points = len(orders)
        
        initial_route = orders
        
        optimized_route = OPT_ROUTES.get(polygon_id)
        
        if optimized_route is None:
            status = "НЕТ ДАННЫХ"
            match = False
        else:
            match = (initial_route == optimized_route)
            status = "СОВПАДАЮТ" if match else "РАЗЛИЧАЮТСЯ"
        
        print(f"\n\nПолигон: {polygon_id}")
        print(f"Точек: {n_points}")
        print(f"\nИсходный маршрут:\n{initial_route}")
        print(f"\nСтоимость исходного маршрута: {calculate_route_cost_arbitrary_order(initial_route, polygon_id)}")
        print(f"\nОптимизированный маршрут:\n{optimized_route}")
        print(f"\nСтоимость оптимизированного маршрута: {calculate_route_cost_arbitrary_order(optimized_route, polygon_id)}")
        print(f"\nРезультат: {status}")

compare_routes_with_initial()

СРАВНЕНИЕ ИСХОДНЫХ И ОПТИМИЗИРОВАННЫХ МАРШРУТОВ


Полигон: AUG--AUG-G-197-1-170-1
Точек: 30

Исходный маршрут:
['AUG-G-197-1-170-1-13371', 'AUG-G-197-1-170-1-16206', 'AUG-G-197-1-170-1-7945', 'AUG-G-197-1-170-1-16128', 'AUG-G-197-1-170-1-3614', 'AUG-G-197-1-170-1-12811', 'AUG-G-197-1-170-1-16157', 'AUG-G-197-1-170-1-1277', 'AUG-G-197-1-170-1-9238', 'AUG-G-197-1-170-1-17683', 'AUG-G-197-1-170-1-7772', 'AUG-G-197-1-170-1-4470', 'AUG-G-197-1-170-1-7769', 'AUG-G-197-1-170-1-6927', 'AUG-G-197-1-170-1-17538', 'AUG-G-197-1-170-1-24021', 'AUG-G-197-1-170-1-22306', 'AUG-G-197-1-170-1-22639', 'AUG-G-197-1-170-1-22383', 'AUG-G-197-1-170-1-23770', 'AUG-G-197-1-170-1-20633', 'AUG-G-197-1-170-1-23694', 'AUG-G-197-1-170-1-21728', 'AUG-G-197-1-170-1-23887', 'AUG-G-197-1-170-1-21232', 'AUG-G-197-1-170-1-23190', 'AUG-G-197-1-170-1-22849', 'AUG-G-197-1-170-1-24154', 'AUG-G-197-1-170-1-21742', 'AUG-G-197-1-170-1-21743']

Стоимость исходного маршрута: 3029.0

Оптимизированный маршрут:
['AUG-G-197-1-170-1-1

In [109]:
print("Время инференса")
for i in EPOS_results.keys():
    print(f"{i}: {EPOS_results[i]['inference_time']}")

Время инференса
AUG--G-158-1-325: 0.005449056625366211
AUG--G-430-2-287: 0.000985860824584961
AUG--G-132-1-71: 0.0032262802124023438
AUG--G-182-3-362: 0.004023075103759766
AUG--AUG-G-344-1-219-7: 0.0051424503326416016
AUG--AUG-G-79-1-37-1: 0.0021729469299316406
AUG--G-216-1-25: 0.0029006004333496094
AUG--AUG-G-197-1-170-1: 0.024868249893188477
AUG--G-92-1-6: 0.0035140514373779297
AUG--G-94-1-125: 0.002538919448852539
